# 4.1 Corner detection and description. Harris and NCC

Images codify valuable information about the environment, which can be retrieved using the appropriate tools. There exist different types of elements of interest that can be extracted such as *corners*, *blobs*, *edges* (our previous chapter), etc., each of them with different properties. These elements are collectively called **image features**. In this chapter we will address one of the most common approaches for extracting information from images, namely **keypoint detection** (i.e. finding *interesting* pixels, or points, in the image). The term *interesting* would depend on the application.

In this notebook we will focus first on detecting **corners**: image points with high variation of intensity in two spatial directions, using the well-known and widely used **Harris corner detector** (<a href="#411">section 4.1.1</a>).

The output of this detector is just the **positions** of the corners in the images. So, in order to be able to distinguish between different corners, we have to incorporate some kind of information about the *appearance* of the keypoint, which will be useful to match corresponding corners in different images. This is called the **description** of the corner. In this sense, we are going to explore how to use a patch around the keypoint as the descriptor, and employ the so-called **Normalized Cross-Correlation** (NCC) to find such matches (<a href="#412">section 4.1.2</a>). Finally, we will put these techniques to work together in <a href="#413">section 4.1.3</a>, that is: detection, description and matching.

Thus, our menu for today includes:

- Harris to detect corners,
- local patches to describe each detected point, and
- NCC (Normalized Cross-Correlation) to match patches.

## Problem context - Image stitching

Image stitching or photo stitching is the process of combining multiple images with overlapping fields of view to produce a segmented panorama or high-resolution image:

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/stitching.jpg" width="600"></center>

<center><i>Fig 1. Example of panorama built using image stitching.</i><br /><br /></center>


Keypoint detection is necessary to find correspondences between images. Robust correspondences are required in order to estimate the necessary transformation to align an image with the image it is being composited on.

Our task in this notebook is to develop the whole image stitching pipeline. **Let's go!**

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Applications of keypoint detection</font>  
> **Suggested prompt**  
> *In the context of computer vision, in which applications, besides image stitching, can keypoint detection in general, and corner detection in particular, be useful?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 04. Keypoint detection'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (20.0, 20.0)

images_path = './images/'

## 4.1.1 Harris corner detector <a id="411" />

The **Harris detector** permits us to **detect corners** in an image: points where the image intensity varies strongly in two directions:

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/corners.png"  width="800"></center>
   $\\[5pt]$

> <font color="#0072B2"><b>🔍 AI Exploration:</b> Intuition Behind Gradient Variations</font>  
> **Suggested prompt**  
> *In computer vision, if I slide a small window over a flat area, an edge, and a corner of a grayscale image, how does the pixel intensity inside the window and their gradients change in each case when shifted in different directions?*

The idea behind the Harris detector is to look for pixels whose local neighborhood changes significantly when it is slightly shifted in different spatial directions.

Let us consider a grayscale image $I$. For a given pixel $(x_0,y_0)$, we define a weighting window $W(x,y)$ centered at that pixel. Then, for a small displacement $(\Delta x,\Delta y)$, we measure the weighted sum of squared intensity differences:

$$
E_{x_0y_0}(\Delta x,\Delta y) =
\sum_{x,y} W(x,y)
\left[
\underbrace{I(x+\Delta x,y+\Delta y)}_{\text{shifted image}}
-
\underbrace{I(x,y)}_{\text{original image}}
\right]^2
$$

where:

- $E_{x_0y_0}(\Delta x,\Delta y)$ is the intensity variation around pixel $(x_0,y_0)$ when the window is shifted by $(\Delta x,\Delta y)$;
- $x$ and $y$ index the pixels involved in the local window;
- $W(x,y)$ is the weighting window centered at $(x_0,y_0)$, usually either binary or Gaussian;
- $I(x,y)$ is the original pixel intensity;
- $I(x+\Delta x,y+\Delta y)$ is the intensity after shifting the image by $(\Delta x,\Delta y)$.

A single large value of $E_{x_0y_0}(\Delta x,\Delta y)$ is not enough to identify a corner: an edge can also produce a large change for some displacement directions. What we need is a compact way to describe how the local window changes for small shifts in different directions.

Computing $E_{x_0y_0}(\Delta x,\Delta y)$ directly for many possible displacements would be expensive. Therefore, we approximate the shifted image using a first-order Taylor expansion:

$$
f(x+\Delta x,y+\Delta y)
\approx
f(x,y)
+
\frac{\partial f}{\partial x}\Delta x
+
\frac{\partial f}{\partial y}\Delta y
$$

Applied to the image $I$, this gives:

$$
I(x+\Delta x,y+\Delta y)
\approx
I(x,y) + I_x\Delta x + I_y\Delta y
$$

where $I_x$ and $I_y$ are the image derivatives at $(x,y)$ along the $x$ and $y$ axes, respectively.

Substituting this approximation into the original expression:

$$
E_{x_0y_0}(\Delta x,\Delta y)
\approx
\sum_{x,y} W(x,y)
\left[
I(x,y) + I_x\Delta x + I_y\Delta y - I(x,y)
\right]^2
$$

The two intensity terms inside the brackets cancel out:

$$
E_{x_0y_0}(\Delta x,\Delta y)
\approx
\sum_{x,y} W(x,y)
\left[
I_x\Delta x + I_y\Delta y
\right]^2
$$

Expanding the square:

$$
E_{x_0y_0}(\Delta x,\Delta y)
\approx
\sum_{x,y} W(x,y)
\left[
I_x^2\Delta x^2 + I_y^2\Delta y^2 + 2I_xI_y\Delta x\Delta y
\right]
$$

This expression can be written in matrix form as:

$$
E_{x_0y_0}(\Delta x,\Delta y)
\approx
\begin{bmatrix}
\Delta x & \Delta y
\end{bmatrix}
M
\begin{bmatrix}
\Delta x \\
\Delta y
\end{bmatrix}
$$

where

$$
M =
\sum_{x,y} W(x,y)
\begin{bmatrix}
I_x^2 & I_xI_y \\
I_xI_y & I_y^2
\end{bmatrix}
$$

The matrix $M$ is computed for each pixel in the image, using the local window centered at $(x_0,y_0)$. It does not store the pixel intensities themselves. Instead, it summarizes how intensity changes inside the local neighborhood.

Each entry of $M$ has a concrete interpretation:

- $M_{xx} = \sum W(x,y)I_x^2$ accumulates the squared derivative values along the $x$ direction inside the window;
- $M_{yy} = \sum W(x,y)I_y^2$ accumulates the squared derivative values along the $y$ direction inside the window;
- $M_{xy} = \sum W(x,y)I_xI_y$ measures how the two derivatives vary together, and is related to the orientation of the local structure.

Thus, $M$ is a compact description of the local intensity-change structure around $(x_0,y_0)$. If the local structure rotates, the individual values inside $M$ also change. This is why the next step is to analyze $M$ through its eigenvectors and eigenvalues, which describe the main directions of change and how strong those changes are.

### Gaining insight into $\mathbf{M}$

Execute the following cells to play with three tiny images — flat region, edge and corner — and observe how $\mathbf{M}$ reacts. No magic yet, just gradients doing their job. The widget also shows the eigenvectors, eigenvalues and Harris response $\mathbf{R}$, which will be used in the next step to decide whether a pixel is a good corner candidate.

In [ ]:
# @title
import numpy as np
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

TOY_IMAGE_SIZE = 21

# -------------------------------------------------------
# Toy images
# -------------------------------------------------------

def make_flat(n=TOY_IMAGE_SIZE):
    return np.full((n, n), 0.5, dtype=np.float32)


def make_edge(n=TOY_IMAGE_SIZE, angle=0, supersampling=16):
    """
    Create an anti-aliased rotated edge using supersampling.

    At angle=0, the right half is white and the left half is black.
    The angle controls the normal direction of the edge.
    """
    N = n * supersampling

    y, x = np.mgrid[0:N, 0:N].astype(np.float32)

    # Coordinates centered at the image center, expressed in low-resolution units
    x = (x + 0.5) / supersampling - n / 2.0
    y = (y + 0.5) / supersampling - n / 2.0

    theta = np.deg2rad(angle)

    # Coordinate along the edge normal
    xr = x * np.cos(theta) + y * np.sin(theta)

    img_hr = np.zeros((N, N), dtype=np.float32)
    img_hr[xr >= 0] = 1.0

    # Downsample to obtain an anti-aliased n x n toy image
    img = cv2.resize(img_hr, (n, n), interpolation=cv2.INTER_AREA)

    return img.astype(np.float32)


def make_corner(n=TOY_IMAGE_SIZE, angle=0, supersampling=16):
    """
    Create an anti-aliased rotated corner using supersampling.

    At angle=0, the bottom-right quadrant is white.
    """
    N = n * supersampling

    y, x = np.mgrid[0:N, 0:N].astype(np.float32)

    # Coordinates centered at the image center, expressed in low-resolution units
    x = (x + 0.5) / supersampling - n / 2.0
    y = (y + 0.5) / supersampling - n / 2.0

    theta = np.deg2rad(angle)

    # Rotated coordinate system
    xr = x * np.cos(theta) + y * np.sin(theta)
    yr = -x * np.sin(theta) + y * np.cos(theta)

    img_hr = np.zeros((N, N), dtype=np.float32)
    img_hr[(xr >= 0) & (yr >= 0)] = 1.0

    # Downsample to obtain an anti-aliased n x n toy image
    img = cv2.resize(img_hr, (n, n), interpolation=cv2.INTER_AREA)

    return img.astype(np.float32)

# -------------------------------------------------------
# Structure tensor M
# -------------------------------------------------------

def compute_M(img, delta_x=1.0, delta_y=1.0, use_gaussian=True, k=0.04):
    img = img.astype(np.float32)

    # Image gradients
    Ix = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)

    # Derivative products
    Ixx = Ix * Ix
    Iyy = Iy * Iy
    Ixy = Ix * Iy

    if use_gaussian:
        n = img.shape[0]
        sigma = max(1.0, n / 6.0)

        # Gaussian weighting window.
        # cv2.getGaussianKernel() is normalized, so W.sum() is approximately 1.
        g = cv2.getGaussianKernel(n, sigma)
        W = g @ g.T
        weighting_name = "Gaussian weighting"
    else:
        # Neighborhood averaging.
        # All pixels in the window contribute with the same weight.
        W = np.ones_like(img, dtype=np.float32) / img.size
        weighting_name = "Neighborhood averaging"

    # Second-moment matrix
    M = np.array([
        [(W * Ixx).sum(), (W * Ixy).sum()],
        [(W * Ixy).sum(), (W * Iyy).sum()]
    ], dtype=np.float32)

    # Local intensity variation for the selected displacement.
    # This is the Taylor/matrix approximation:
    # E ≈ [Δx Δy] M [Δx Δy]^T
    displacement = np.array([delta_x, delta_y], dtype=np.float32)
    E_delta = float(displacement.T @ M @ displacement)

    # M is symmetric, so eigh is appropriate
    eigvals, eigvecs = np.linalg.eigh(M)

    # Sort eigenvalues/eigenvectors from largest to smallest
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # Harris response
    det_M = np.linalg.det(M)
    trace_M = np.trace(M)
    R = det_M - k * (trace_M ** 2)

    return M, E_delta, eigvals, eigvecs, R, weighting_name

# -------------------------------------------------------
# Helper: draw eigenvector arrows
# -------------------------------------------------------

def draw_eigenvector_arrows(ax, eigvals, eigvecs, n, max_eig):
    cx = (n - 1) / 2.0
    cy = (n - 1) / 2.0

    colors = ["tab:red", "tab:cyan"]
    labels = [r"$\lambda_1$", r"$\lambda_2$"]

    for i in range(2):
        if max_eig <= 1e-8 or eigvals[i] <= 1e-8:
            continue

        vx = eigvecs[0, i]
        vy = eigvecs[1, i]

        # Only for display stability: v and -v represent the same direction
        if vx < 0 or (abs(vx) < 1e-8 and vy < 0):
            vx, vy = -vx, -vy

        # Arrow length proportional to the square root of the eigenvalue
        length = 0.38 * n * np.sqrt(eigvals[i] / max_eig)

        dx = length * vx
        dy = length * vy

        ax.annotate(
            "",
            xy=(cx + dx, cy + dy),
            xytext=(cx, cy),
            arrowprops=dict(
                arrowstyle="-|>",
                color=colors[i],
                lw=2,
                mutation_scale=18
            )
        )

        ax.text(
            cx + dx * 1.12,
            cy + dy * 1.12,
            f"{labels[i]}={eigvals[i]:.2f}",
            color=colors[i],
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="center"
        )

    # Mark centroid
    ax.plot(cx, cy, "wo", markersize=5)

# -------------------------------------------------------
# Visualization
# -------------------------------------------------------

def show_tensor_demo(angle=0, delta_x=1.0, delta_y=1.0, use_gaussian=True, k=0.04):
    n = TOY_IMAGE_SIZE

    flat = make_flat(n)
    edge = make_edge(n, angle)
    corner = make_corner(n, angle)

    examples = [
        ("Flat region", flat),
        (f"Edge ({angle}°)", edge),
        (f"Corner ({angle}°)", corner),
    ]

    results = []

    for title, img in examples:
        M, E_delta, eigvals, eigvecs, R, weighting_name = compute_M(
            img,
            delta_x=delta_x,
            delta_y=delta_y,
            use_gaussian=use_gaussian,
            k=k
        )
        results.append((title, img, M, E_delta, eigvals, eigvecs, R))

    max_eig = max(eigvals[0] for _, _, _, _, eigvals, _, _ in results)

    fig, axes = plt.subplots(2, 3, figsize=(11, 7.7))

    fig.suptitle(
        f"Window operation: {weighting_name}    |    "
        f"Displacement: Δx={delta_x:.1f}, Δy={delta_y:.1f}",
        fontsize=13
    )

    for col, (title, img, M, E_delta, eigvals, eigvecs, R) in enumerate(results):

        # ---------------------------------------------------
        # Top row: toy image + eigenvector arrows
        # ---------------------------------------------------
        axes[0, col].imshow(img, cmap="gray", vmin=0, vmax=1)
        axes[0, col].set_title(title)
        axes[0, col].axis("off")

        draw_eigenvector_arrows(
            axes[0, col],
            eigvals=eigvals,
            eigvecs=eigvecs,
            n=n,
            max_eig=max_eig
        )

        # ---------------------------------------------------
        # Bottom row: matrix M
        # ---------------------------------------------------
        axes[1, col].imshow(M, cmap="viridis")
        axes[1, col].set_title("M", pad=10)

        axes[1, col].set_xticks([0, 1])
        axes[1, col].set_yticks([0, 1])
        axes[1, col].set_xticklabels(["x", "y"])
        axes[1, col].set_yticklabels(["x", "y"])

        for i in range(2):
            for j in range(2):
                axes[1, col].text(
                    j, i, f"{M[i, j]:.2f}",
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=11,
                    fontweight="bold"
                )

        # Text below the matrix
        axes[1, col].text(
            0.5, -0.18,
            f"E(Δx,Δy)≈{E_delta:.2f}\n"
            f"λ1={eigvals[0]:.2f}, λ2={eigvals[1]:.2f}\n"
            f"R={R:.2f}",
            transform=axes[1, col].transAxes,
            ha="center",
            va="top",
            fontsize=11
        )

    plt.subplots_adjust(hspace=0.28, wspace=0.25, bottom=0.15, top=0.88)
    plt.show()

    print("Interpretation:")
    print(f"- Current operation: {weighting_name}.")
    print("- Gaussian weighting: pixels near the center of the window have more influence.")
    print("- Neighborhood averaging: all pixels in the window have the same influence.")
    print(f"- E(Δx, Δy) is computed for Δx={delta_x:.1f}, Δy={delta_y:.1f}.")
    print("- Rotating the pattern changes the entries of M,")
    print("  but the eigenvalue pattern still reveals the type of local structure.")

# -------------------------------------------------------
# Interactive widget
# -------------------------------------------------------

widgets.interact(
    show_tensor_demo,
    angle=widgets.ToggleButtons(
        options=[0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
        value=0,
        description="Rotate:"
    ),
    delta_x=widgets.FloatSlider(
        value=1.0,
        min=-3.0,
        max=3.0,
        step=0.5,
        description="Δx:"
    ),
    delta_y=widgets.FloatSlider(
        value=1.0,
        min=-3.0,
        max=3.0,
        step=0.5,
        description="Δy:"
    ),
    use_gaussian=widgets.Checkbox(
        value=True,
        description="Gaussian window"
    ),
    k=widgets.FloatSlider(
        value=0.04,
        min=0.02,
        max=0.10,
        step=0.01,
        description="k:"
    )
);

## Achieving rotation invariance

In the previous example, you saw that the numerical entries of $M$ change when the local structure rotates. This is expected: $M$ is expressed using the image axes $x$ and $y$.

However, the local structure itself has not changed. An edge is still an edge after rotation, and a corner is still a corner. What changes is how that structure is represented with respect to the image coordinate system.

Since $M$ is a symmetric matrix, it has two orthogonal eigenvectors $v_1$ and $v_2$:

$$
M = VDV^T
$$

where

$$
D =
\begin{bmatrix}
\lambda_1 & 0 \\
0 & \lambda_2
\end{bmatrix}
$$

and

$$
V =
\begin{bmatrix}
v_{1x} & v_{2x} \\
v_{1y} & v_{2y}
\end{bmatrix}
$$

The eigenvectors $v_1$ and $v_2$ define the **principal directions of intensity variation** inside the window. The eigenvalues $\lambda_1$ and $\lambda_2$ indicate how strong the variation is along those directions.

In other words, diagonalizing $M$ means changing from the original image axes $(x,y)$ to a local coordinate system aligned with the dominant intensity changes. This is the key idea behind the rotation robustness of the Harris detector: rotation changes the orientation of the eigenvectors, but not the eigenvalue pattern.

The key idea is:

- the entries of $M$ change when the patch rotates;
- the eigenvectors rotate with the patch;
- the eigenvalues remain the same for the same local structure, only rotated.

Therefore, Harris does not need to know the absolute orientation of the edge or corner. It only needs to know how strong the intensity changes are along the two principal directions. This gives the usual interpretation:

<br /><center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/harris_eigenvalues_map.PNG" width="650">
    <br > <i>Fig 4. Relation between eigenvalues and region types.</i></center>



### The R variable

However, **computing eigenvalues and eigenvectors is a costly process**, and we need to do it for each pixel in the image! Thus, Harris proposed an alternative metric to get the same information. It is based on the definition of a scalar variable $R$ that indexes the same domain:

$$R = \underbrace{\lambda_1 \lambda_2}_\text{Determinant} - k (\underbrace{\lambda_1+\lambda_2}_\text{Trace})^2$$

and given that the trace and the determinant of $\mathbf{M}$ and $\mathbf{D}$ are the same, then:

$$R = \text{det}(\mathbf{M}) - k (\text{trace}(\mathbf{M}))^2$$

so **there is no need to compute the eigenvalues!** In these equations, $k$ is an empirically determined constant value in the range $[0.04,0.06]$.

This $R$ value is interpreted as follows:
- $R$ is large and positive at corners,
- $R$ is negative at edges, and
- $R$ is small at flat regions.

### The Harris detector properties

It has a number of appealing features:

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/harris_detector_properties.PNG" width="850px">
    <br ></center>

A corner that is visible at one scale may disappear or look different at another scale.



## Finally: the Harris detector algorithm

To summarize, the Harris detector follows these steps:

1. Compute the image derivatives $I_x$ and $I_y$, for example using Sobel filters.

2. Compute the gradient products: $I_x^2$, $I_y^2$, $I_xI_y$

3. Accumulate these products inside a local window, usually using Gaussian weighting (this is equivalent to applying Gaussian smoothing) or neighbor averaging, which additionally provides **robustness to noise**. This gives the matrix $\mathbf{M}$ at each pixel:

   $$
   M =
   \sum_{(x,y)\in W}
   w(x,y)
   \begin{bmatrix}
   I_x^2 & I_xI_y \\
   I_xI_y & I_y^2
   \end{bmatrix}
   $$

4. Compute the Harris response image $\mathbf{R}$:

   $$
   R = \det(M) - k \cdot \operatorname{trace}(M)^2
   $$

5. Keep pixels whose response is high enough:

   $$
   R > \text{threshold}
   $$

6. Select local maxima to avoid multiple nearby detections for the same corner.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/Harris_algorithm.PNG">
    <br ></center>

### Let's start the party! Initial data

Let's start by **displaying our testing image** `car.jpeg` in the RGB color space. This is the image that we are going to use to familiarize with the concepts behind the Harris corner detector. Remember that by default OpenCV reads the image in BGR mode, so you'll have to convert it.

In [ ]:
# Read 'car.jpeg' image, convert to RGB and display it
# Write your code here!

# Read image
image = cv2.imread(images_path + 'car.jpeg')
image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

# Show it
plt.imshow(image);

### **<span style="color:green"><b><i>ASSIGNMENT 1a: R matrix</i></b></span>**

Compute the $\mathbf{R}$ matrix of the previous `car.jpeg` image using the Harris detector from OpenCV and display it. Take into account that the **Harris method** (and most keypoint detectors) **works with the grayscale version of the input images**. The color image will be used just for visualization purposes.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Using OpenCV Harris Response</font>  
> In practice, the Harris detector is already implemented in the OpenCV function cv2.cornerHarris(), so let's see how it works!

> **Suggested prompt**  
> *I need to use OpenCV to compute the Harris response through the `cv2.cornerHarris()` function. Do not write the full solution. Explain what its main parameters mean, and what output I should expect.*

In [ ]:
def apply_harris(image, size_window, size_sobel, const_k):
    """
    Apply the Harris detector to an RGB image.
    """
    # Convert to gray
    gray = cv2.cvtColor(None, None)
    gray = np.float32(gray)

    # Apply Harris
    harris_response = cv2.cornerHarris(None, None, None, None)

    return harris_response

# Give it a try!
size_window = 3
size_sobel = 3
const_k = 0.04
harris = apply_harris(image, size_window, size_sobel, const_k)
plt.imshow(None, None);

If you plot the $\mathbf{R}$ matrix, also called response map, you should see an image where borders are black pixels, corners are white ones, and flat surfaces are gray.

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

**It is interesting** to play a bit with the input parameters. Then **answer the following questions**:

- What happens if the size of $W$ is 1?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What happens if the size of $W$ is large? (*note that `cv2.cornerHarris()` uses neighborhood averaging instead of Gaussian filtering*)
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

### Thresholding the response map $\mathbf{R}$

The previous code computes the response map $\mathbf{R}$ but, in the end, we need information about which image points contain corners, that is, a binary yes/no response. This is typically done by setting a threshold so pixels with a response higher than it are considered corners.  

### **<span style="color:green"><b><i>ASSIGNMENT 1b: Analyzing $\mathbf{R}$</i></b></span>**

In order to make an informed choice of that threshold, print the maximum and minimum values in the response map $\mathbf{R}$, and take a look at how their positive values are distributed.

In [ ]:
# Assignment 1b
#
# Analyze the distribution of Harris response values
#
# Print max and min values of the R matrix (given)
print("Max: " , None, "\nMin: ", None)

# Remove extreme values only for visualization
harris_values = harris.ravel()
harris_positive = harris_values[harris_values > 0]

print("Max:", harris_values.max())
print("Positive values:", len(harris_positive))

plt.figure(figsize=(8, 4))
plt.hist(harris_positive, bins=100)
plt.title("Distribution of positive Harris response values")
plt.xlabel("Harris response value")
plt.ylabel("Number of pixels")
plt.show()

Are those values the same for different images or input parameters?  

**No**, they are not. As they change, it is necessary to define the threshold relative to the maximum value in $\mathbf{R}$ (in this way we skip normalization).  

### **<span style="color:green"><b><i>ASSIGNMENT 1c: Thresholding</i></b></span>**

Use the OpenCV's [`cv2.threshold()`](https://docs.opencv.org/trunk/d7/d1b/group__imgproc__misc.html#gae8a4a146d1ca78c626a53577199e9c57) method to binarize the $\mathbf{R}$ matrix using a threshold that you think is suitable. Then, display the obtained results.

*Hint: For computing the threshold, we'll multiply the maximum value by a scale factor. If you do `1*harris.max()` (a scale factor of 1), only the pixels having a R response equal to `harris.max()` will be set to white. If you do `0.5*harris.max()` (a scale factor of 0.5), those having a R response equal or higher than half `harris.max()` will be set to white, and so on!*

In [ ]:
# Assignment 1c
# Define the threshold wrt the maximum value, apply binarization to the image and display the result
# Write your code here!

# Define the threshold wrt the maximum value
threshold = None*harris.max()

# Apply binarization and show the result
_, corners = cv2.threshold(None, None, None, None)
plt.imshow(None, None);

### Non-maximum suppression

The resulting binarized image shows white points where the $\mathbf{R}$ values are over the threshold and black points otherwise. However, around maximum values, there are many points that are close to each other. That is, we are getting multiple responses for the same corner.

The general solution to this is to apply what is called **non-maximum suppression** to those clusters of points, keeping only a point for each of them (remember that this step was also part of the Canny edge detector!).$\\[10pt]$

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/nonmax.png" width="600"><br />
    <i>Fig 6. An illustrative example of how non-maximum suppression works.</i></center>$\\[5pt]$

To make this easier for you, we provide to you a `nonmaxsuppts()` function with an implementation of the non-maximum suppression step, which finds local maxima as corners within a window. This method returns two lists of numbers with the rows `r` and columns `c`, respectively, of the isolated corners.

In [ ]:
# This method has been provided to you
from scipy import signal
def nonmaxsuppts(cim, radius, thresh):
    """ Binarize and apply non-maximum suppression.

        Args:
            cim: the harris 'R' image
            radius: the aperture size of local maxima window
            thresh: the threshold value for binarization

        Returns:
            r, c: two numpy vectors being the row (r) and the column (c) of each keypoint
    """

    rows, cols = np.shape(cim)
    sze = 2 * radius + 1
    mx = signal.order_filter(cim, np.ones([sze, sze]), sze ** 2 - 1)
    bordermask = np.zeros([rows, cols]);
    bordermask[radius:(rows - radius), radius:(cols - radius)] = 1
    cim = np.array(cim)
    r, c = np.where((cim == mx) & (cim > thresh) & (bordermask == 1))
    return r, c

### **<span style="color:green"><b><i>ASSIGNMENT 1d: Non-maxima suppression</i></b></span>**

Apply non-maxima suppression to the previous $\mathbf{R}$ image and draw the detected keypoints on the image.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Working with keypoints</font>  
> In order to represent keypoints, OpenCV defines a set of functions for drawing objects related to feature detection. Let's explore one of them. *Note that the utilization of these functions implies the transformation of `r` and `c` to a `cv2.KeyPoint` list.*

> **Suggested prompt**  
> *In the context of feature detection in OpenCV, explain the main parameters of the `cv2.drawKeypoints()` function and what output I should expect. Then, explain what a `cv2.KeyPoint` object represents. Do not write a full solution.*

In [ ]:
# Assignment 1d
# Apply non-max suppression to R and display the remaining keypoints on the image
# Write your code here!

# Define the radius of the keypoint neighborhood to be considered
radius_non_maxima = 5

# Apply non maxima suppression
r,c = nonmaxsuppts(None, None, threshold)

# Convert coordinates to cv2.KeyPoint list
kps = [cv2.KeyPoint(None, None, None) for i in range(len(r))]

# Draw keypoints
image_corners = np.copy(image)
cv2.drawKeypoints(None, None, None)

# Show image with corners
plt.imshow(image_corners);

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Choosing a Harris threshold</font>  
> **Suggested prompt**  
> *I have selected a threshold for the Harris response using a percentage of `harris.max()`. Review my reasoning. Help me check whether my choice is supported by the response map, the histogram and the detected points. Do not choose a threshold for me. Then, briefly explain other common thresholding strategies.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

## 4.1.2 Keypoint matching through NCC

After detecting keypoints, we only know **where** they are in the image. This may be enough for some tasks, such as drawing corners, but it is not enough if we want to match points between two images. To match keypoints, we need to **describe** what the image looks like around each one.

A simple way to do this is to take a small image patch centered at the keypoint. Then, instead of comparing only two pixel coordinates, we compare two small neighborhoods.

The question is:

> How similar are these two patches?

A first idea would be to compare the raw pixel intensities directly. However, this is not very robust. The same patch may look brighter, darker, or have stronger contrast in another image.

Normalized Cross-Correlation, or **NCC**, solves this by comparing patches after removing two simple effects:

- **brightness**, by subtracting the mean intensity of each patch;
- **contrast**, by dividing by the norm of each zero-mean patch.

So, before comparing two patches, NCC transforms them into normalized patches:

$$
\hat{f} =
\frac{f - \bar{f}}{\|f - \bar{f}\|}
\qquad
\hat{h} =
\frac{h - \bar{h}}{\|h - \bar{h}\|}
$$

where $f$ and $h$ are two image patches, and $\bar{f}$ and $\bar{h}$ are their mean intensities. The denominator measures the magnitude of the intensity variations after removing the mean brightness. It is proportional to the patch contrast: a high-contrast patch has a larger denominator, while a low-contrast patch has a smaller one.

Once both patches are normalized, NCC compares them using a dot product:

$$
NCC_{f,h} =
\sum_{(i,j)\in W}
\hat{f}(i,j)\hat{h}(i,j)
$$

Equivalently, if we reshape each patch as a vector, NCC can be interpreted as:

$$
NCC_{f,h} = \hat{f}^{T}\hat{h} = \cos (\theta)
$$

That is, NCC measures the cosine of the angle between two normalized patch vectors.

This gives a very useful intuition:

- if the two patches are very similar, their vectors point in almost the same direction, so $NCC$ is close to $1$;
- if they are unrelated, $NCC$ is closer to $0$;
- if they are opposite, $NCC$ can be close to $-1$.

Therefore, high NCC values indicate that two patches are visually similar, even if one is brighter or has a different contrast. The following figure illustrates this:

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/ncc_intuition_and_examples.png">
    <br ></center>

When the patch $h$ is used as a template and compared against different positions of an image $f$, NCC can be written as:

$$
NCC(x,y) =
\sum_{(i,j)\in W}
\hat{f}(x+i,y+j)\hat{h}(i,j)
$$

Here, the template patch $h$ is compared with the image patch of $f$ centered at position $(x,y)$. The positions with higher NCC values are the most similar to the template.

> <font color="#0072B2"><b>🔍 AI Exploration:</b> NCC vs correlation</font>  
> **Suggested prompt**  
> *In the context of computer vision, what is the relationship between Normalized Cross-Correlation and convolution?*

The next cell of code shows two patches from two different images, and a template. Then, it illustrates how cross correlation and normalized cross correlation are computed.

In [ ]:
im_patch_1 = np.array([[220,30,200],[210,205,220],[220,40,230]]);
im_patch_2 = np.array([[210,230,220],[20,215,10],[10,250,30]]);
template = np.array([[255,0,255],[255,255,255],[255,0,255]]);

plt.figure(figsize=(8,4))
plt.subplot(1,3,1), plt.xticks([]), plt.yticks([])
plt.title('Patch from image 1')
plt.imshow(im_patch_1, cmap='gray');
plt.subplot(1,3,2), plt.xticks([]), plt.yticks([])
plt.title('Patch from image 2')
plt.imshow(im_patch_2, cmap='gray');
plt.subplot(1,3,3)
plt.title('Template'), plt.xticks([]), plt.yticks([])
plt.imshow(template, cmap='gray');
plt.show()

# Correlation
corr_1 = np.sum(im_patch_1 * template)
corr_2 = np.sum(im_patch_2 * template)
print('Cross correlation between patch from image 1 and template:', corr_1)
print('Cross correlation between patch from image 2 and template:', corr_2)

# Normalized correlation
im_patch_1_mean = np.mean(im_patch_1)
im_patch_2_mean = np.mean(im_patch_2)
template_mean = np.mean(template)

im_patch_1_contrast = np.sqrt(np.sum((im_patch_1 - im_patch_1_mean)**2))
im_patch_2_contrast = np.sqrt(np.sum((im_patch_2 - im_patch_2_mean)**2))
template_contrast = np.sqrt(np.sum((template - template_mean)**2))

ncc_1 = np.sum(((im_patch_1-im_patch_1_mean)/im_patch_1_contrast)*((template-template_mean)/template_contrast))
ncc_2 = np.sum(((im_patch_2-im_patch_2_mean)/im_patch_2_contrast)*((template-template_mean)/template_contrast))
print('Normalized cross correlation between patch from image 1 and template:', ncc_1)
print('Normalized cross correlation between patch from image 2 and template:', ncc_2)

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Computing NCC</font>  
> **Suggested prompt**  
> *I need to use OpenCV to compute the NCC of an image and a template with the `cv2.matchTemplate()` function. Do not write the full solution. Explain what its main parameters mean (propose `cv2.TM_CCOEFF_NORMED`), and what output I should expect.*

The next code will help you to understand how NCC works.   
- **Review it and try to explain what is done here** (you can modify `p_index` and `w_temp` parameters).

In [ ]:
# This snippet has been provided to you. Play with 'p_index' and 'w_temp' values

# Index of the focusing point
p_index = 50

# NCC window aperture
w_temp = 20

# Get the NCC window
p_r, p_c = r[p_index], c[p_index]
template = image[p_r-w_temp:p_r+w_temp+1,p_c-w_temp:p_c+w_temp+1]

# Calculate the NCC image (note the cv2.TM_CCOEFF_NORMED parameter!)
ncc = cv2.matchTemplate(image, template, cv2.TM_CCOEFF_NORMED)

# Search the maximum in NCC
p_x, p_y = p_c, p_r
[max_y, max_x] = np.where(ncc == np.amax(ncc))

# Print some interesting information
print('> Coordinates of [max_y,max_x]:', max_y, max_x)
print('NCC value:', ncc[max_y,max_x])
print('> Other random coordinates. e.g. [100] [300]')
print('NCC value:', ncc[100,300])

# Draw a rectangle in matching points
image_draw = np.copy(image)
ncc_draw = np.copy(ncc)

cv2.rectangle(image_draw,(p_x-w_temp,p_y-w_temp),(p_x+w_temp+1,p_y+w_temp+1),(0,255,0),3)
cv2.rectangle(ncc_draw,(int(max_x-w_temp),int(max_y-w_temp)),(int(max_x+w_temp+1),int(max_y+w_temp+1)),0,3)

# Show the original image
plt.subplot(121)
plt.imshow(image_draw)
plt.title('Original image')

# Show the NCC image
plt.subplot(122)
plt.imshow(ncc_draw, cmap='gray')
plt.title('NCC')
plt.show()

As you can see, the patch with the green rectangle in the left image has a maximum NCC value at the black rectangle in the right image. As you can imagine, since the template is extracted from the same image, the NCC value will be 1 at the same exact position (the template and the image at that point are exactly the same!). But this is just an example to illustrate how all this works.

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now, **answer the following questions**:

- Describe what variables `p_index` and `w_temp` do.
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What does the NCC image/response represent?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What would happen if `template` was taken from a similar (but not the same) image?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    

## 4.1.3 Time to work! Find matches between images with (some) overlapping <a id="413" />

It is time to apply what has been learned during this notebook. Coming back to image stitching, **you are asked** to compute matches between two images with overlapping areas. The images are `park_l.jpeg` (left) and `park_r.jpeg` (right).

**For that, you should use the Harris corner detector and the NCC descriptor**. Also, you will have to draw the keypoints and the matches using [`cv2.drawMatches()`](https://docs.opencv.org/3.4/d4/d5d/group__features2d__draw.html#ga7421b3941617d7267e3f2311582f49e1), which takes two images, two lists of `cv2.KeyPoint` objects and a list of [`cv2.DMatch`](https://docs.opencv.org/3.4/d4/de0/classcv_1_1DMatch.html) objects as input. To create a `cv2.DMatch` object it is used `cv2.DMatch(idx_l, idx_r,1)`, being `idx_l` and `idx_r` the indices of the matching keypoints (in the left and right lists of keypoints, respectively).

*Tip: [Drawing functions documentation](https://docs.opencv.org/master/d4/d5d/group__features2d__draw.html)*

Results should look something like this (output of [`cv2.drawMatches()`](https://docs.opencv.org/3.4/d4/d5d/group__features2d__draw.html#ga7421b3941617d7267e3f2311582f49e1)):

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/keypoint_detection/images/index.png" width="1000">

### **<span style="color:green"><b><i>ASSIGNMENT 2a: First, detect keypoints with Harris</i></b></span>**

- Use the Harris corner detector to get the keypoints of both grayscale images (do not forget to apply non-max suppression to the resulting $\mathbf{R}$ image!)
- Convert them to two lists of `cv2.KeyPoint` objects.

In [ ]:
# Assignment 2a
# Detect Harris keypoints in the images and convert them to a 'cv2.KeyPoint' list

# Write your code here!

# Size of Sobel kernel
size_sobel = 3

# Size of Gaussian window (in openCV neighbor averaging)
size_window = 2

# Empiric constant K
const_k = 0.05

# Read images
image_l = cv2.imread(images_path + 'park_l.jpeg')
image_r = cv2.imread(images_path + 'park_r.jpeg')
image_l = cv2.cvtColor(image_l,cv2.COLOR_BGR2RGB)
image_r = cv2.cvtColor(image_r,cv2.COLOR_BGR2RGB)

# Get gray images
image_l_gray = cv2.cvtColor(None, None)
image_r_gray = cv2.cvtColor(None, None)

# Compute corners
harris_l = cv2.cornerHarris(None, None, None, None)
harris_r = cv2.cornerHarris(None, None, None, None)

# For non-max-supp
radius_non_maxima = 3

# Apply non maximum suppression
r_l,c_l = nonmaxsuppts(None, None, 0.1*harris_l.max())
r_r,c_r = nonmaxsuppts(None, None, 0.1*harris_r.max())

# Convert coordinates to cv2.KeyPoint list
kps_l = [cv2.KeyPoint(None, None, None) for i in range(len(r_l))]
kps_r = [cv2.KeyPoint(None, None, None) for i in range(len(r_r))]

### **<span style="color:green"><b><i>ASSIGNMENT 2b: Avoiding issues</i></b></span>**

Now, we will address an intermediate step to avoid future issues. As we will need to extract a template around the keypoints and it might happen that those can be detected too close to the border (e.g. a keypoint found at (0,0)), we will add a *padding* to the image using the OpenCV's method
[`cv2.copyMakeBorder()`](https://docs.opencv.org/2.4/modules/imgproc/doc/filtering.html?highlight=copymakeborder#void%20copyMakeBorder(InputArray%20src,%20OutputArray%20dst,%20int%20top,%20int%20bottom,%20int%20left,%20int%20right,%20int%20borderType,%20const%20Scalar&%20value)) (`cv2.BORDER_REFLECT` is the best padding option in this case)

In [ ]:
# Assignment 2b
# Add padding to the images
# Write your code here!

# Aperture size of the NCC window
w_temp = 20

# Apply border reflect padding to images
image_l_pad = cv2.copyMakeBorder(None, w_temp, w_temp, w_temp, w_temp, None)
image_r_pad = cv2.copyMakeBorder(None, w_temp, w_temp, w_temp, w_temp, None)

### **<span style="color:green"><b><i>ASSIGNMENT 2c: Matching</i></b></span>**

And finally:
- For every $i$-th keypoint in the left image:
    1. Take a template around that point.
    2. Compute the NCC value using that template with **the whole right image**, as in the provided example.
    3. Since we want to retrieve true matches, find **the maximum NCC value** at the positions of the keypoints in the right image.
    4. If the maximum NCC value (for a $j$ keypoint in the right image) is greater than a certain threshold (e.g. 0.65, but tune it for robustness), create a `cv2.DMatch` object with the indexes $(i,j)$ of the keypoints and add it to a list of matches.
- Once all matches are defined, use the `cv2.drawMatches()` method to display the resulting color image with the matches.

*Tip: When displaying the resulting image, it is preferable to make a copy of the original color image and use it as the output image for the `cv2.drawMatches()` method. This way it keeps the same size as the original one.*

Be careful with the number of keypoints detected, as **it could take a while to process all of them!**

In [ ]:
# Initialize matches list
matches = []

# For each keypoint in left image
for p_index in range(len(r_l)):

    # 1. Get the template (note the w_temp offset because of the padding in previous step)
    p_r, p_c = r_l[p_index]+w_temp, c_l[p_index]+w_temp
    template = image_l_pad[p_r-None:p_r+None+1,p_c-None:p_c+None+1]

    # 2. Compute NCC of the left keypoint in right image
    ncc = cv2.matchTemplate(None, None, None)

    # 3. Find max value in NCC (only at the right keypoints)
    max_value = np.amax(ncc[None, None])

    # 4. If match is good
    if max_value > 0.65:
        # Include in match list
        max_index = np.argmax(None)
        matches.append((p_index, max_index))

# Cast matches list to cv2.DMatch list
matches = np.asarray(matches)
matches = [cv2.DMatch(int(matches[None,None]), int(matches[None,None]),1) for i in range(matches.shape[0])]

# Draw matches
image_matches = np.copy(image_l)
image_matches = cv2.drawMatches(None, None, None, None, None,image_matches)

# And show them
plt.imshow(image_matches);

Finally! You have detected and matched keypoints in two overlapping images, let's analyze them!

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Improving keypoint matching</font>  
> **Suggested prompt**  
> *I have matched keypoints in two images using local patches and NCC. Help me reflect on the quality of the matches and how the result could be improved. Ask me to analyze correct matches (inliers), wrong matches (outliers), possible causes of failure, and strategies to reduce outliers. Do not give me the final conclusion directly; guide me so that I can write it myself.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

> 🚀 <font color="#7B1FA2"><b>AI Knowledge Transfer:</b> From keypoints to image stitching</font>  
> In this final activity, you will use an AI assistant to transfer what you have learned in this notebook to a new task: **image stitching**.
>
> Your goal is not to copy a ready-made prompt, but to **design your own prompt** so that the AI assistant helps you build two small prototypes.
>
> **Part A — Using your own pipeline**
>
> Design a prompt asking the AI assistant to help you create a simple image stitching prototype using:
>
> - the keypoints already detected in this notebook;
> - your own functions for patch extraction;
> - NCC for patch comparison;
> - matched keypoints;
> - a simple image alignment;
> - a basic stitched image.
>
> Your prompt must clearly state that OpenCV automatic stitching functions are **not allowed** in this first version.
>
> **Part B — Using OpenCV**
>
> Then, design a second prompt asking the AI assistant to solve the same task using OpenCV's high-level stitching API:
>
> ```python
> cv2.Stitcher_create()
> ```
>
> **Your prompts should include**
>
> - the context of the task;
> - what variables, functions or keypoints you already have;
> - what the assistant is allowed to use;
> - what the assistant is not allowed to use;
> - what output you expect;
> - a request for short, understandable code.
>
> Before asking for the final code, ask the AI assistant to briefly explain the proposed steps.

### Your prompt — Part A: Using your own pipeline

```text
Write your prompt here.
```

In [ ]:
# Your code for Part A here!

### Your prompt — Part B: Using OpenCV

```text
Write your prompt here.
```

In [ ]:
# Your code for Part B here!

## Conclusion

This was hard work, but finally you made it! In this notebook you have learned to:

- detect corners using the Harris detector,
- match corners using image patches and NCC, and
- develop the first steps for an image stitching system.

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>